In [1]:
import numpy as np
import matplotlib.pyplot as plt

import pandas as pd
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder,StandardScaler,TargetEncoder,FunctionTransformer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import classification_report,confusion_matrix,roc_curve

In [2]:
df = pd.read_csv('telco-churn-dataset.csv') # Load the dataset

In [3]:
df.head(3)

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes


## Necessary Preprocessing 

In [4]:
# Convert TotalCharges to numeric
df = df[df['TotalCharges'] != ' ']
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='raise')

# Also convert SeniorCitizen column to numeric
df['SeniorCitizen'] = pd.to_numeric(df['SeniorCitizen'],errors='raise')

# Map gender before hand to avoid data leakage and jargaon
df['gender'] = df['gender'].map({'Male':1,"Female":0})

# Lets seperate columns into group so we can proceed easily
# Dropped churn as it will be handled seprately and gender need's a different mapping
# Also SeniorCitizen is numeric

drop_cols = ['MonthlyCharges','TotalCharges','customerID']
target_encoding_cols = ['InternetService','PaymentMethod']
ordinal_cols = ['Contract']


In [5]:
X , y = df.drop(columns=['Churn']),df['Churn'].map( lambda x : 1 if x == 'Yes' else 0)

# We seperate the columns after seperating to avoid data leak
# The rest of the cols are just to be mapped 
service_columns = [
    x for x in X.columns 
    if x not in target_encoding_cols + ordinal_cols + drop_cols + ['gender', 'SeniorCitizen', 'tenure']
]

X_train , X_test , y_train , y_test = train_test_split(
                                                            X,y,
                                                            test_size = 0.2,
                                                            stratify=y,
                                                            random_state = 42
)


In [6]:
# Make sure to pass the slice itself cause if otherwise there could be data leaks in case we try to pass the original dataframe.
def to_binary_map(X_sliced):
    col_map = {
                'Yes': 1,
                'No': 0,
                'No internet service': 0,
                'No phone service': 0
    }
    return X_sliced.apply( lambda x : x.map(col_map).fillna(0) ) # This IDK why somehow caused data leak

 # Also wrtting our map function inside the FunctionTransformer
binary_transformer = FunctionTransformer(to_binary_map)

## Building the Pipeline

In [7]:
preprocessor = ColumnTransformer(
                                    transformers = [
                                                        ('binary_encoder',binary_transformer,service_columns),
                                                        ('ordinal_encoder',OrdinalEncoder(),ordinal_cols),
                                                        ('target_encoder',TargetEncoder(smooth=10.0),target_encoding_cols),
                                                        ('drop_columns','drop',drop_cols)
                                    ],
                            remainder = 'passthrough'
)

In [8]:
rf_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(max_depth=6, class_weight='balanced', random_state=42))
])

rf_pipeline.fit(X_train, y_train)
print(f"Random Forest Train Score: {rf_pipeline.score(X_train, y_train):.4f}")
print(f"Random Forest Test Score: {rf_pipeline.score(X_test, y_test):.4f}")

Random Forest Train Score: 0.7639
Random Forest Test Score: 0.7292


In [9]:
# The pipelining was successfully , actually I dont know how so I isolated the features , map , variables and pipeline as much as I can to create a leak free pipeline
# Now finetuning ... 